# InternVL3 P2.1 confirmed source preparation

Owner-only, Internet ON, accelerator NONE. Run only after audit PASS, ChatGPT checks and Standard Merge. The old 'InternVL3 Source + Model Prep FIX v2' output remains the model input; this notebook downloads only repository source, never the 4.18 GB weights or runtime. It accepts exact confirmed final main merge SHA and verifies effective v4 before packaging full Git history. Draft F2 is rejected. Save the output for the offline notebook.

In [ ]:
# Fill these release facts only AFTER independent audit PASS, ChatGPT code/audit
# comparison, Standard Merge and ChatGPT verification of merge parents/tree/main.
# Input settings are release metadata; do not edit any production source.
CONFIRMATION = {
    "final_merge_sha": "",
    "independent_audit": "PENDING",
    "chatgpt_code_audit": "PENDING",
    "research_lead_confirmed": False,
    "chatgpt_merge_main_verification": "PENDING",
    "audit_evidence_reference": "",
    "merge_evidence_reference": "",
}


In [ ]:
import json
from pathlib import Path
import subprocess
import sys

sha = CONFIRMATION["final_merge_sha"]
if len(sha) != 40 or any(c not in "0123456789abcdef" for c in sha):
    raise RuntimeError("Research Lead must supply exact confirmed final merge SHA")
for field in ("independent_audit", "chatgpt_code_audit", "chatgpt_merge_main_verification"):
    if CONFIRMATION[field] != "PASS":
        raise RuntimeError("External review PASS required: " + field)
if CONFIRMATION["research_lead_confirmed"] is not True:
    raise RuntimeError("Research Lead release confirmation required")
if not all(CONFIRMATION[k].strip() for k in ("audit_evidence_reference", "merge_evidence_reference")):
    raise RuntimeError("Record audit and post-merge verification references")
work = Path.cwd()
repo = work / "p21_confirmed_source"
if repo.exists():
    raise RuntimeError("Fresh source preparation required; refusing overwrite")
# Full history and a credential-free public URL. No model/runtime downloads.
subprocess.run(["git", "clone", "--no-checkout", "https://github.com/tantdna2/SafeShift.git", str(repo)], check=True)
def git(*args):
    return subprocess.check_output(["git", "-C", str(repo), *args]).decode().strip()
if git("rev-parse", "refs/remotes/origin/main") != sha:
    raise RuntimeError("Confirmed final merge must equal fetched main")
parents = git("rev-list", "--parents", "-n", "1", sha).split()
if len(parents) != 3 or parents[1] != "f0b5ea775b3c5bbe5e8618ba1372e59b8b8e150f":
    raise RuntimeError("Exact Standard Merge parent1 BASE required; Draft F2 forbidden")
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", sha], check=True)
sys.path.insert(0, str(repo))
from safeshift.runners.p21_kaggle import prepare_source
manifest = prepare_source(repo, CONFIRMATION, work / "p21_source_output")
print(json.dumps(manifest, indent=2, sort_keys=True))
print("Save this notebook output and attach it to the offline shard0 notebook.")


The source release archive and `p21_source_release.json` are the new output. Attach them to `w2_internvl3_p21_shard0_kaggle.ipynb` together with the two existing FIX v2 outputs, the original dataset/manifest/provenance, and the exact historical result manifest **and status**. No GPU, dataset or model is read by this source-only notebook.